In [1]:
"""
TCGA-BRCA multi-omics: filtered cohort + 3-way imputation cohort, OVERALL SURVIVAL.

Settings produced:
  - filtered                     : patients with all 3 modalities present
  - imputation_filteredpart      : full-cohort training, eval on patients with all 3 modalities present
  - imputation_extrapart         : full-cohort training, eval on patients with >=1 missing modality
  - imputation_overall           : full-cohort training, eval on full test set

Method families:
  Filtered setting:
    benchmarks      : 3 single-modality + early fusion (one head each)
    late_fusion     : independent per-modality, all 6 ensembles (option 1 style)
    metafusion      : rho_search (no ablation)
    joint marginal  : with all 6 ensembles + cohort
    joint shapley   : with all 6 ensembles + cohort

  Imputation 3-way setting:
    benchmarks            : trained on full cohort with sentinels (3-way eval)
    late_fusion           : independent per-modality on all rows with sentinels (option 1, naive)
    late_fusion_avail     : per-modality trained ONLY on rows where modality is available
                            test-time fusion uses only available modalities per patient (option 2)
    metafusion rho_search : trained on full cohort, evaluated 3-way
    joint marginal        : with missing_value handling, evaluated 3-way
    joint shapley         : with missing_value handling, evaluated 3-way

What is NOT included:
  - No KNN real-imputation setting
  - No meta-fusion ablation (rho=0)

Inputs:
  ./tcga/tcga_brca_processed/filtered_cohort.npz
  ./tcga/tcga_brca_processed/full_cohort.npz       (must be generated -- see notes at bottom)
  ./tcga/tcga_brca_raw/TCGA-BRCA.survival.tsv.gz

Outputs (in OUTPUT_DIR):
  tcga_brca_survival_4settings_20reps_raw.csv
  tcga_brca_survival_4settings_20reps_avg_with_se.csv
  tcga_brca_survival_4settings_partial.csv  (saved after every rep)
"""

import os
import copy
import random
from typing import List, Dict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

from meta_fusion.utils import *
from meta_fusion.models import *
from meta_fusion.methods import *
from meta_fusion.methodsextra_new import *
from meta_fusion.benchmarks import *


# ============================================================
# USER SETTINGS
# ============================================================

DATA_DIR = "./tcga/tcga_brca_processed"
RAW_DATA_DIR = "./tcga/tcga_brca_raw"
OUTPUT_DIR = "./tcga_brca_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NPZ_FILE_FILTERED = "filtered_cohort.npz"
NPZ_FILE_FULL = "full_cohort.npz"
SURVIVAL_FILE = "TCGA-BRCA.survival.tsv.gz"

MISSING_VALUE = -999.0
SURVIVAL_THRESHOLD_DAYS = 5 * 365
NUM_CLASSES = 2

RANDOM_STATE = 42
USE_GPU = torch.cuda.is_available()
TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TRAIN = 0.20
BATCH_SIZE = 64
NUM_REPETITIONS = 20
REPETITION_SEEDS = [RANDOM_STATE + i for i in range(NUM_REPETITIONS)]
EPOCHS = 100

HIDDEN_DIMS = [256, 128]

BASE_CONFIG = {
    "task_type": "classification",
    "output_dim": NUM_CLASSES,
    "use_gpu": USE_GPU,
    "rho_list": [0, 0.1, 0.5],
    "rho_list_ncl": [0, 0.1, 0.5],
    "epochs": EPOCHS,
    "init_lr": 1e-3,
    "weight_decay": 1e-4,
    "gamma": 1.0,
    "divergence_weight_type": "uniform",
    "burn_in_epochs": 0,
    "optimal_k": 2,
    "divergence_weight_scale": 1.0,
    "ensemble_methods": [
        "simple_average",
        "weighted_average",
        "majority_voting",
        "weighted_voting",
        "greedy_ensemble",
        "meta_learner",
    ],
    "epochs_meta_learner": 20,
    "progress": False,
    "random_state": RANDOM_STATE,
    "verbose": True,
    "ckpt_dir": os.path.join(OUTPUT_DIR, f"checkpoints_seed_{RANDOM_STATE}"),
}


# ============================================================
# HELPERS
# ============================================================


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_config_for_seed(split_seed: int):
    config = copy.deepcopy(BASE_CONFIG)
    config["random_state"] = split_seed
    config["ckpt_dir"] = os.path.join(OUTPUT_DIR, f"checkpoints_split_seed_{split_seed}")
    os.makedirs(config["ckpt_dir"], exist_ok=True)
    return config


def parse_tcga_patient_id(barcode: str) -> str:
    parts = str(barcode).split("-")
    if len(parts) < 3:
        return None
    return "-".join(parts[:3])


def get_modality_fully_missing_mask(a: np.ndarray, missing_value: float) -> np.ndarray:
    return np.all(a == missing_value, axis=1)


def get_no_missingness_mask(arrays: List[np.ndarray], missing_value: float) -> np.ndarray:
    keep = np.ones(arrays[0].shape[0], dtype=bool)
    for a in arrays:
        keep &= ~get_modality_fully_missing_mask(a, missing_value)
    return keep


# ============================================================
# DATASET
# ============================================================


class TCGATensorDataset(Dataset):
    def __init__(self, arrays: List[np.ndarray], y: np.ndarray):
        self.arrays = [torch.tensor(a, dtype=torch.float32) for a in arrays]
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (*[a[idx] for a in self.arrays], self.y[idx])


class SingleModalityTensorDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


# ============================================================
# DATA LOADING (filtered + full, both with survival)
# ============================================================


def _load_survival_table(raw_dir: str):
    """Load survival.tsv.gz. Return DataFrame indexed by patient_id with
    OS event/time columns standardized."""
    surv_path = os.path.join(raw_dir, SURVIVAL_FILE)
    if not os.path.exists(surv_path):
        raise FileNotFoundError(f"Survival file not found: {surv_path}")
    surv = pd.read_csv(surv_path, sep="\t", compression="gzip", low_memory=False)

    id_col = None
    for cand in ["sample", "_PATIENT", "submitter_id", "bcr_patient_barcode"]:
        if cand in surv.columns:
            id_col = cand
            break
    if id_col is None:
        id_col = surv.columns[0]

    os_event_col = None
    os_time_col = None
    for cand in ["OS", "OS_status", "os_event"]:
        if cand in surv.columns:
            os_event_col = cand
            break
    for cand in ["OS.time", "OS_time", "os_days", "OS_days"]:
        if cand in surv.columns:
            os_time_col = cand
            break
    if os_event_col is None or os_time_col is None:
        raise RuntimeError(
            f"Could not find OS columns. Available: {list(surv.columns)}"
        )

    surv["_patient_id"] = surv[id_col].astype(str).apply(parse_tcga_patient_id)
    surv = surv.dropna(subset=["_patient_id"])
    surv[os_event_col] = pd.to_numeric(surv[os_event_col], errors="coerce")
    surv[os_time_col] = pd.to_numeric(surv[os_time_col], errors="coerce")
    surv = surv.dropna(subset=[os_event_col, os_time_col])
    surv = surv.drop_duplicates(subset=["_patient_id"], keep="first").set_index("_patient_id")

    return surv, os_event_col, os_time_col


def _apply_survival_threshold(patient_ids, surv, os_event_col, os_time_col, threshold_days):
    """For each patient, return:
       y_i = 1 if died <= threshold; 0 if alive past threshold; -1 if drop.
    """
    y = np.full(len(patient_ids), -1, dtype=np.int64)
    n_no_surv = 0
    n_censored_early = 0
    for i, pid in enumerate(patient_ids):
        if pid not in surv.index:
            n_no_surv += 1
            continue
        os_event = int(surv.loc[pid, os_event_col])
        os_time = float(surv.loc[pid, os_time_col])
        if os_event == 1 and os_time <= threshold_days:
            y[i] = 1
        elif os_time > threshold_days:
            y[i] = 0
        else:
            n_censored_early += 1
    return y, n_no_surv, n_censored_early


def load_filtered_cohort_with_survival(data_dir, raw_dir, threshold_days):
    """Filtered cohort: all 3 modalities present + valid survival label."""
    print("=" * 80)
    print("Loading FILTERED cohort + survival")
    print("=" * 80)
    npz_path = os.path.join(data_dir, NPZ_FILE_FILTERED)
    if not os.path.exists(npz_path):
        raise FileNotFoundError(f"Not found: {npz_path}. Run tcga_brca_preprocess.py first.")
    d = np.load(npz_path, allow_pickle=True)
    patient_ids = np.array([str(p) for p in d["patient_ids"]])
    arrays = [d["m0_mrna"].astype(np.float32),
              d["m1_mirna"].astype(np.float32),
              d["m2_methyl"].astype(np.float32)]
    print(f"  filtered cohort (PAM50 step): {len(patient_ids)} patients")

    surv, ev_col, t_col = _load_survival_table(raw_dir)
    y, n_no, n_cens = _apply_survival_threshold(patient_ids, surv, ev_col, t_col, threshold_days)
    keep = (y >= 0)
    print(f"  drop: no_survival={n_no}, censored_early={n_cens}, keep={keep.sum()}")
    patient_ids = patient_ids[keep]
    y = y[keep]
    arrays = [a[keep] for a in arrays]

    n0 = int((y == 0).sum())
    n1 = int((y == 1).sum())
    print(f"  Filtered final: {len(y)} patients  (alive={n0} {100*n0/len(y):.1f}%, "
          f"died={n1} {100*n1/len(y):.1f}%)")
    print(f"  shapes: mRNA {arrays[0].shape}, miRNA {arrays[1].shape}, methyl {arrays[2].shape}")
    return patient_ids, y, arrays


def load_full_cohort_with_survival(data_dir, raw_dir, threshold_days):
    """Full cohort: at least 1 modality present, sentinel-marked missingness +
    valid survival label."""
    print("=" * 80)
    print("Loading FULL cohort + survival")
    print("=" * 80)
    npz_path = os.path.join(data_dir, NPZ_FILE_FULL)
    if not os.path.exists(npz_path):
        raise FileNotFoundError(
            f"Not found: {npz_path}. You must re-run tcga_brca_preprocess.py with the "
            f"FULL COHORT block enabled (see notes in this script's docstring)."
        )
    d = np.load(npz_path, allow_pickle=True)
    patient_ids = np.array([str(p) for p in d["patient_ids"]])
    arrays = [d["m0_mrna"].astype(np.float32),
              d["m1_mirna"].astype(np.float32),
              d["m2_methyl"].astype(np.float32)]
    print(f"  full cohort (PAM50 step): {len(patient_ids)} patients")

    surv, ev_col, t_col = _load_survival_table(raw_dir)
    y, n_no, n_cens = _apply_survival_threshold(patient_ids, surv, ev_col, t_col, threshold_days)
    keep = (y >= 0)
    print(f"  drop: no_survival={n_no}, censored_early={n_cens}, keep={keep.sum()}")
    patient_ids = patient_ids[keep]
    y = y[keep]
    arrays = [a[keep] for a in arrays]

    # Print missingness pattern
    miss_per_mod = [int(get_modality_fully_missing_mask(a, MISSING_VALUE).sum()) for a in arrays]
    print(f"  missing rows per modality: mRNA={miss_per_mod[0]}, "
          f"miRNA={miss_per_mod[1]}, methyl={miss_per_mod[2]}")
    n0 = int((y == 0).sum())
    n1 = int((y == 1).sum())
    print(f"  Full final: {len(y)} patients  (alive={n0} {100*n0/len(y):.1f}%, "
          f"died={n1} {100*n1/len(y):.1f}%)")
    return patient_ids, y, arrays


# ============================================================
# SPLITTING
# ============================================================


def build_patient_level_splits(patient_ids, y, test_size, val_size_within_train, random_state):
    """Stratified train/val/test at patient level.
    With one row per patient, this is just row-level."""
    unique_patients, first_idx = np.unique(patient_ids, return_index=True)
    unique_labels = y[first_idx]

    p_train, p_test = train_test_split(
        unique_patients, test_size=test_size,
        random_state=random_state, stratify=unique_labels,
    )
    train_label_lookup = dict(zip(unique_patients, unique_labels))
    train_labels = np.array([train_label_lookup[p] for p in p_train])
    p_train, p_val = train_test_split(
        p_train, test_size=val_size_within_train,
        random_state=random_state, stratify=train_labels,
    )
    train_idx = np.where(np.isin(patient_ids, p_train))[0]
    val_idx = np.where(np.isin(patient_ids, p_val))[0]
    test_idx = np.where(np.isin(patient_ids, p_test))[0]
    return train_idx, val_idx, test_idx


def build_imputation_splits(full_patient_ids, full_y, full_arrays, missing_value,
                            test_size, val_size_within_train, random_state):
    """For the imputation 3-way setting on the full cohort.

    Splits filtered (no missingness) and extra (some missingness) patients
    separately with stratified splits, then concatenates.
    """
    filtered_mask = get_no_missingness_mask(full_arrays, missing_value)
    filtered_idx = np.where(filtered_mask)[0]
    extra_idx = np.where(~filtered_mask)[0]

    print(f"  full cohort breakdown: filtered (all 3 mods) = {len(filtered_idx)}, "
          f"extra (>=1 missing) = {len(extra_idx)}")

    # Stratified split on filtered subset
    f_pids = full_patient_ids[filtered_idx]
    f_y = full_y[filtered_idx]
    f_train, f_val, f_test = build_patient_level_splits(
        f_pids, f_y, test_size, val_size_within_train, random_state,
    )
    f_train_global = filtered_idx[f_train]
    f_val_global = filtered_idx[f_val]
    f_test_global = filtered_idx[f_test]

    # Stratified split on extra subset
    e_pids = full_patient_ids[extra_idx]
    e_y = full_y[extra_idx]
    if len(extra_idx) > 0:
        e_train, e_val, e_test = build_patient_level_splits(
            e_pids, e_y, test_size, val_size_within_train, random_state,
        )
        e_train_global = extra_idx[e_train]
        e_val_global = extra_idx[e_val]
        e_test_global = extra_idx[e_test]
    else:
        e_train_global = np.array([], dtype=int)
        e_val_global = np.array([], dtype=int)
        e_test_global = np.array([], dtype=int)

    imp_train = np.sort(np.concatenate([f_train_global, e_train_global]))
    imp_val = np.sort(np.concatenate([f_val_global, e_val_global]))
    imp_test = np.sort(np.concatenate([f_test_global, e_test_global]))

    test_filtered_mask = np.isin(imp_test, f_test_global)
    test_extra_mask = np.isin(imp_test, e_test_global)

    return {
        "imp_train_idx": imp_train,
        "imp_val_idx": imp_val,
        "imp_test_idx": imp_test,
        "core_test_idx": f_test_global,
        "extra_test_idx": e_test_global,
        "test_filtered_mask": test_filtered_mask,
        "test_extra_mask": test_extra_mask,
    }


def subset_by_indices(arrays, y, idx):
    return [a[idx] for a in arrays], y[idx]


# ============================================================
# STANDARDIZATION
# ============================================================


def standardize_per_modality(train_arrays, val_arrays, test_arrays):
    """Filtered-cohort version: no sentinels. Fit on train, transform val/test."""
    sc_tr, sc_v, sc_te = [], [], []
    for xtr, xv, xte in zip(train_arrays, val_arrays, test_arrays):
        scaler = StandardScaler()
        sc_tr.append(scaler.fit_transform(xtr).astype(np.float32))
        sc_v.append(scaler.transform(xv).astype(np.float32))
        sc_te.append(scaler.transform(xte).astype(np.float32))
    return sc_tr, sc_v, sc_te


def standardize_per_modality_with_sentinel(train_arrays, val_arrays, test_arrays, missing_value):
    """Imputation version: fit scaler only on train rows where modality is present.
    Apply to non-missing rows in val/test. Sentinel rows preserved as missing_value."""
    sc_tr, sc_v, sc_te = [], [], []
    for xtr, xv, xte in zip(train_arrays, val_arrays, test_arrays):
        train_avail = ~get_modality_fully_missing_mask(xtr, missing_value)
        if train_avail.sum() == 0:
            sc_tr.append(xtr.astype(np.float32))
            sc_v.append(xv.astype(np.float32))
            sc_te.append(xte.astype(np.float32))
            continue
        scaler = StandardScaler()
        scaler.fit(xtr[train_avail])

        def transform(x):
            x_out = x.copy()
            avail = ~get_modality_fully_missing_mask(x, missing_value)
            if avail.any():
                x_out[avail] = scaler.transform(x[avail])
            # Sentinel rows kept as-is (still equal to missing_value).
            return x_out.astype(np.float32)

        sc_tr.append(transform(xtr))
        sc_v.append(transform(xv))
        sc_te.append(transform(xte))
    return sc_tr, sc_v, sc_te


# ============================================================
# LOADERS
# ============================================================


def make_loaders_from_arrays(train_arrays, val_arrays, test_arrays,
                             y_train, y_val, y_test, batch_size):
    train_ds = TCGATensorDataset(train_arrays, y_train)
    val_ds = TCGATensorDataset(val_arrays, y_val)
    test_ds = TCGATensorDataset(test_arrays, y_test)
    drop_last = (len(train_ds) >= 2 * batch_size)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=drop_last)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)
    return train_loader, val_loader, test_loader


def make_single_modality_loader(x, y, batch_size, shuffle, drop_last):
    ds = SingleModalityTensorDataset(x, y)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)


def subset_test_loader(arrays, y, mask, batch_size):
    ds = TCGATensorDataset([a[mask] for a in arrays], y[mask])
    return DataLoader(ds, batch_size=batch_size, shuffle=False, drop_last=False)


def make_partitioned_test_loaders(setting, batch_size):
    """For a setting dict with test_arrays, y_test, test_filtered_mask, test_extra_mask:
    return (full_loader, filtered_part_loader, extra_part_loader)."""
    full_test_loader = DataLoader(
        TCGATensorDataset(setting["test_arrays"], setting["y_test"]),
        batch_size=batch_size, shuffle=False, drop_last=False,
    )
    filtered_test_loader = subset_test_loader(
        setting["test_arrays"], setting["y_test"],
        setting["test_filtered_mask"], batch_size,
    )
    extra_test_loader = subset_test_loader(
        setting["test_arrays"], setting["y_test"],
        setting["test_extra_mask"], batch_size,
    )
    return full_test_loader, filtered_test_loader, extra_test_loader


# ============================================================
# METRICS (binary-aware)
# ============================================================


def multiclass_sensitivity(y_true, y_pred, num_classes):
    vals = []
    for c in range(num_classes):
        yt = (y_true == c).astype(int)
        yp = (y_pred == c).astype(int)
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        vals.append(tp / (tp + fn) if (tp + fn) > 0 else np.nan)
    return float(np.nanmean(vals))


def multiclass_specificity(y_true, y_pred, num_classes):
    vals = []
    for c in range(num_classes):
        yt = (y_true == c).astype(int)
        yp = (y_pred == c).astype(int)
        tn, fp, fn, tp = confusion_matrix(yt, yp, labels=[0, 1]).ravel()
        vals.append(tn / (tn + fp) if (tn + fp) > 0 else np.nan)
    return float(np.nanmean(vals))


def safe_auc(y_true, y_prob):
    try:
        if len(np.unique(y_true)) < 2:
            return float("nan")
        if y_prob.ndim == 2 and y_prob.shape[1] == 2:
            return float(roc_auc_score(y_true, y_prob[:, 1]))
        return float(roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro"))
    except Exception:
        return float("nan")


def safe_multiclass_auc(y_true, y_prob):
    return safe_auc(y_true, y_prob)


def compute_classification_metrics(y_true, y_pred, y_prob, num_classes=NUM_CLASSES):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro")),
        "auc_ovr": safe_auc(y_true, y_prob),
        "sensitivity": multiclass_sensitivity(y_true, y_pred, num_classes),
        "specificity": multiclass_specificity(y_true, y_pred, num_classes),
    }


# Patch into meta_fusion submodules
import meta_fusion.methodsextra_new as mf_extra
import meta_fusion.benchmarks as mf_benchmarks

mf_extra.multiclass_sensitivity = multiclass_sensitivity
mf_extra.multiclass_specificity = multiclass_specificity
mf_extra.safe_multiclass_auc = safe_multiclass_auc
mf_extra.compute_classification_metrics = compute_classification_metrics
mf_benchmarks.multiclass_sensitivity = multiclass_sensitivity
mf_benchmarks.multiclass_specificity = multiclass_specificity
mf_benchmarks.safe_multiclass_auc = safe_multiclass_auc
mf_benchmarks.compute_classification_metrics = compute_classification_metrics


# ============================================================
# BENCHMARK MODELS
# ============================================================


class BenchmarkSingleModalityModel(nn.Module):
    def __init__(self, modality_idx, input_dim, hidden_dims, output_dim):
        super().__init__()
        self.modality_idx = modality_idx
        self.model = MLP_Net(input_dim, hidden_dims, output_dim)

    def forward(self, modalities):
        return self.model(modalities[self.modality_idx])


class BenchmarkEarlyFusionModel(nn.Module):
    def __init__(self, input_dims, hidden_dims, output_dim):
        super().__init__()
        self.model = MLP_Net(sum(input_dims), hidden_dims, output_dim)

    def forward(self, modalities):
        return self.model(torch.cat(modalities, dim=1))


def build_benchmark_models(input_dims):
    return [
        BenchmarkSingleModalityModel(0, int(input_dims[0]), HIDDEN_DIMS, NUM_CLASSES),
        BenchmarkSingleModalityModel(1, int(input_dims[1]), HIDDEN_DIMS, NUM_CLASSES),
        BenchmarkSingleModalityModel(2, int(input_dims[2]), HIDDEN_DIMS, NUM_CLASSES),
        BenchmarkEarlyFusionModel(input_dims, HIDDEN_DIMS, NUM_CLASSES),
    ]


def _mlp(input_dim):
    return MLP_Net(int(input_dim), HIDDEN_DIMS, NUM_CLASSES)


def build_models(input_dims):
    return [_mlp(d) for d in input_dims]


# ============================================================
# LATE FUSION ALL-ENSEMBLES (option 1: trains on all rows including sentinels)
# ============================================================


class LateFusionAllEnsemblesBenchmark:
    """Independent per-modality training on all rows (including sentinel-marked).
    Test-time fusion uses all per-modality outputs (option 1, naive baseline)."""

    def __init__(self, config, models):
        self.config = copy.deepcopy(config)
        self.models = models
        self.model_num = len(models)
        self.num_classes = int(config["output_dim"])
        self.use_gpu = bool(config["use_gpu"])
        self.device = torch.device("cuda" if self.use_gpu and torch.cuda.is_available() else "cpu")
        self.epochs = int(config["epochs"])
        self.lr = float(config["init_lr"])
        self.weight_decay = float(config["weight_decay"])
        for m in self.models:
            m.to(self.device)
        self.optimizers = [
            optim.Adam(m.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            for m in self.models
        ]
        self.criterion = nn.CrossEntropyLoss()
        self.best_val_task_losses = [float("inf")] * self.model_num
        self.ens_idxs = list(range(self.model_num))
        self.ensemble_methods = [
            "simple_average", "weighted_average",
            "majority_voting", "weighted_voting",
            "best_single", "greedy_ensemble",
        ]

    def train(self, train_loader, val_loader):
        for mi in range(self.model_num):
            best_val_loss = float("inf")
            best_state = copy.deepcopy(self.models[mi].state_dict())
            for epoch in range(self.epochs):
                self.models[mi].train()
                for batch in train_loader:
                    modalities, target = batch[:-1], batch[-1]
                    if self.use_gpu:
                        modalities = [x.cuda() for x in modalities]
                        target = target.cuda()
                    self.optimizers[mi].zero_grad()
                    logits = self.models[mi](modalities[mi])
                    loss = self.criterion(logits, target)
                    loss.backward()
                    self.optimizers[mi].step()
                vl = self._eval_one_model(mi, val_loader)
                if vl < best_val_loss:
                    best_val_loss = vl
                    best_state = copy.deepcopy(self.models[mi].state_dict())
            self.models[mi].load_state_dict(best_state)
            self.best_val_task_losses[mi] = best_val_loss
        self.ens_idxs = self._greedy_forward_selection_on_val(val_loader)

    def _eval_one_model(self, mi, loader):
        self.models[mi].eval()
        total_loss = 0.0
        total_n = 0
        with torch.no_grad():
            for batch in loader:
                modalities, target = batch[:-1], batch[-1]
                if self.use_gpu:
                    modalities = [x.cuda() for x in modalities]
                    target = target.cuda()
                logits = self.models[mi](modalities[mi])
                loss = self.criterion(logits, target)
                bs = target.size(0)
                total_loss += float(loss.item()) * bs
                total_n += bs
        return total_loss / total_n if total_n > 0 else float("inf")

    def _greedy_forward_selection_on_val(self, val_loader):
        for m in self.models:
            m.eval()
        per_model_probs = [[] for _ in range(self.model_num)]
        all_targets = []
        with torch.no_grad():
            for batch in val_loader:
                modalities, target = batch[:-1], batch[-1]
                if self.use_gpu:
                    modalities = [x.cuda() for x in modalities]
                    target = target.cuda()
                for i, model in enumerate(self.models):
                    out = model(modalities[i])
                    per_model_probs[i].append(F.softmax(out, dim=1).cpu().numpy())
                all_targets.append(target.cpu().numpy())
        per_model_probs = [np.concatenate(p) for p in per_model_probs]
        all_targets = np.concatenate(all_targets)
        sorted_models = sorted(range(self.model_num),
                               key=lambda i: self.best_val_task_losses[i])
        chosen = [sorted_models[0]]
        best_acc = (np.argmax(per_model_probs[chosen[0]], axis=1) == all_targets).mean()
        for cand in sorted_models[1:]:
            trial = chosen + [cand]
            avg_probs = np.mean([per_model_probs[i] for i in trial], axis=0)
            acc = (np.argmax(avg_probs, axis=1) == all_targets).mean()
            if acc > best_acc:
                chosen = trial
                best_acc = acc
        return chosen

    def _weights_inverse_loss(self):
        eps = 1e-8
        inv = np.array([1.0 / (l + eps) for l in self.best_val_task_losses], dtype=np.float32)
        inv = inv / inv.sum()
        return torch.tensor(inv, dtype=torch.float32, device=self.device)

    def test(self, test_loader, missing_value=None):
        # missing_value arg ignored: option 1 uses all per-modality outputs always.
        records = {m: {"y_true": [], "y_pred": [], "y_prob": []} for m in self.ensemble_methods}
        cohort_records = [{"y_true": [], "y_pred": [], "y_prob": []} for _ in range(self.model_num)]
        for m in self.models:
            m.eval()
        weights_all = self._weights_inverse_loss()
        with torch.no_grad():
            for batch in test_loader:
                modalities, target = batch[:-1], batch[-1]
                if self.use_gpu:
                    modalities = [x.cuda() for x in modalities]
                    target = target.cuda()
                outputs = []
                for i, model in enumerate(self.models):
                    out = model(modalities[i])
                    outputs.append(out)
                    prob = torch.softmax(out, dim=1).cpu().numpy()
                    pred = torch.argmax(out, dim=1).cpu().numpy()
                    cohort_records[i]["y_true"].append(target.cpu().numpy())
                    cohort_records[i]["y_pred"].append(pred)
                    cohort_records[i]["y_prob"].append(prob)
                outputs_stack = torch.stack(outputs)

                for method in self.ensemble_methods:
                    if method == "simple_average":
                        final_output = torch.mean(outputs_stack, dim=0)
                    elif method == "weighted_average":
                        final_output = torch.sum(
                            weights_all.unsqueeze(1).unsqueeze(2) * outputs_stack, dim=0)
                    elif method == "majority_voting":
                        final_pred = torch.mode(outputs_stack.argmax(dim=2), dim=0).values
                        final_output = F.one_hot(final_pred, num_classes=self.num_classes).float()
                    elif method == "weighted_voting":
                        top_preds = torch.argmax(outputs_stack, dim=2)
                        bs = outputs_stack.shape[1]
                        wv = torch.zeros((bs, self.num_classes), device=self.device)
                        for k, mp in enumerate(top_preds):
                            wv.scatter_add_(1, mp.unsqueeze(1),
                                torch.full((bs, 1), float(weights_all[k]), device=self.device))
                        final_output = wv
                    elif method == "best_single":
                        best_idx = self.best_val_task_losses.index(min(self.best_val_task_losses))
                        final_output = outputs_stack[best_idx]
                    elif method == "greedy_ensemble":
                        ens_w = weights_all[self.ens_idxs]
                        ens_w = ens_w / ens_w.sum()
                        final_output = torch.sum(
                            ens_w.unsqueeze(1).unsqueeze(2) * outputs_stack[self.ens_idxs], dim=0)
                    else:
                        raise ValueError(method)
                    prob = torch.softmax(final_output, dim=1).cpu().numpy()
                    pred = torch.argmax(final_output, dim=1).cpu().numpy()
                    records[method]["y_true"].append(target.cpu().numpy())
                    records[method]["y_pred"].append(pred)
                    records[method]["y_prob"].append(prob)

        results = {}
        for method, rec in records.items():
            yt = np.concatenate(rec["y_true"])
            yp = np.concatenate(rec["y_pred"])
            yprob = np.concatenate(rec["y_prob"])
            results[method] = compute_classification_metrics(yt, yp, yprob,
                                                              num_classes=self.num_classes)
        results["cohort"] = []
        for rec in cohort_records:
            yt = np.concatenate(rec["y_true"])
            yp = np.concatenate(rec["y_pred"])
            yprob = np.concatenate(rec["y_prob"])
            results["cohort"].append(
                compute_classification_metrics(yt, yp, yprob, num_classes=self.num_classes))
        return results


# ============================================================
# LATE FUSION AVAILABLE-ONLY (option 2: trains per-modality on available rows;
# test-time fusion uses only available modalities per patient)
# ============================================================


class LateFusionAvailableOnlyBenchmark:
    """
    Option 2 baseline:
      - For each modality, train ONLY on rows where that modality is available.
      - Test-time fusion uses only the modalities that are available for each
        sample (pattern grouping).
    All 6 ensemble methods.
    """

    def __init__(self, config, input_dims):
        self.config = copy.deepcopy(config)
        self.input_dims = input_dims
        self.model_num = len(input_dims)
        self.num_classes = int(self.config["output_dim"])
        self.use_gpu = bool(self.config["use_gpu"])
        self.device = torch.device("cuda" if self.use_gpu and torch.cuda.is_available() else "cpu")
        self.missing_value = MISSING_VALUE

        self.models = [
            MLP_Net(int(d), HIDDEN_DIMS, self.num_classes).to(self.device)
            for d in input_dims
        ]
        self.optimizers = [
            optim.Adam(m.parameters(),
                       lr=self.config["init_lr"], weight_decay=self.config["weight_decay"])
            for m in self.models
        ]
        self.criterion = nn.CrossEntropyLoss()
        self.ensemble_methods = [
            "simple_average", "weighted_average",
            "majority_voting", "weighted_voting",
            "best_single", "greedy_ensemble",
        ]
        self.best_val_task_losses = [float("inf")] * self.model_num
        self.ens_idxs = list(range(self.model_num))

    def _make_available_loaders_one_modality(self, train_arrays, val_arrays, y_train, y_val, mi):
        train_mask = ~get_modality_fully_missing_mask(train_arrays[mi], self.missing_value)
        val_mask = ~get_modality_fully_missing_mask(val_arrays[mi], self.missing_value)
        x_train = train_arrays[mi][train_mask]
        y_train_mod = y_train[train_mask]
        x_val = val_arrays[mi][val_mask]
        y_val_mod = y_val[val_mask]
        train_loader = make_single_modality_loader(
            x_train, y_train_mod, batch_size=BATCH_SIZE,
            shuffle=True, drop_last=(len(y_train_mod) >= BATCH_SIZE),
        )
        val_loader = make_single_modality_loader(
            x_val, y_val_mod, batch_size=BATCH_SIZE,
            shuffle=False, drop_last=False,
        )
        return train_loader, val_loader, int(train_mask.sum()), int(val_mask.sum())

    def train(self, train_arrays, val_arrays, y_train, y_val):
        for mi in range(self.model_num):
            tl, vl, n_tr, n_v = self._make_available_loaders_one_modality(
                train_arrays, val_arrays, y_train, y_val, mi
            )
            print(f"  modality {mi}: avail train n={n_tr}, val n={n_v}")
            if n_tr == 0:
                self.best_val_task_losses[mi] = float("inf")
                continue
            model = self.models[mi]
            optimizer = self.optimizers[mi]
            best_state = copy.deepcopy(model.state_dict())
            best_val_loss = float("inf")
            for epoch in range(int(self.config["epochs"])):
                model.train()
                for xb, yb in tl:
                    xb = xb.to(self.device)
                    yb = yb.to(self.device)
                    optimizer.zero_grad()
                    logits = model(xb)
                    loss = self.criterion(logits, yb)
                    loss.backward()
                    optimizer.step()
                v_loss = self._evaluate_single_model_loss(model, vl)
                if v_loss < best_val_loss:
                    best_val_loss = v_loss
                    best_state = copy.deepcopy(model.state_dict())
            model.load_state_dict(best_state)
            self.best_val_task_losses[mi] = best_val_loss

        finite = [(i, l) for i, l in enumerate(self.best_val_task_losses) if np.isfinite(l)]
        finite_sorted = sorted(finite, key=lambda x: x[1])
        self.ens_idxs = [i for i, _ in finite_sorted]

    def _evaluate_single_model_loss(self, model, loader):
        if len(loader.dataset) == 0:
            return float("inf")
        model.eval()
        total_loss = 0.0
        total_n = 0
        with torch.no_grad():
            for xb, yb in loader:
                xb = xb.to(self.device)
                yb = yb.to(self.device)
                logits = model(xb)
                loss = self.criterion(logits, yb)
                bs = yb.size(0)
                total_loss += float(loss.item()) * bs
                total_n += bs
        return total_loss / total_n if total_n > 0 else float("inf")

    def _group_indices_by_availability(self, modalities, missing_value):
        bs = modalities[0].shape[0]
        availability = []
        for m in modalities:
            is_missing = torch.all(m == missing_value, dim=1)
            availability.append(~is_missing)
        availability = torch.stack(availability, dim=1)
        patterns = {}
        for i in range(bs):
            pat = tuple(bool(x.item()) for x in availability[i])
            patterns.setdefault(pat, []).append(i)
        for k in patterns:
            patterns[k] = torch.tensor(patterns[k], dtype=torch.long,
                                        device=modalities[0].device)
        return patterns

    def _get_weights(self, present_models):
        valid = [(m, self.best_val_task_losses[m]) for m in present_models
                 if np.isfinite(self.best_val_task_losses[m])]
        if len(valid) == 0:
            return None
        eps = 1e-8
        inv = np.array([1.0 / (l + eps) for _, l in valid], dtype=np.float32)
        inv = inv / inv.sum()
        m_order = [m for m, _ in valid]
        wmap = {m: float(w) for m, w in zip(m_order, inv)}
        weights = np.array([wmap[m] for m in present_models], dtype=np.float32)
        weights = weights / weights.sum()
        return torch.tensor(weights, dtype=torch.float32, device=self.device)

    def test(self, test_loader, missing_value=None):
        if missing_value is None:
            missing_value = self.missing_value
        records = {m: {"y_true": [], "y_pred": [], "y_prob": []} for m in self.ensemble_methods}
        cohort_records = [{"y_true": [], "y_pred": [], "y_prob": []} for _ in range(self.model_num)]
        for model in self.models:
            model.eval()
        with torch.no_grad():
            for batch in test_loader:
                modalities, target = batch[:-1], batch[-1]
                if self.use_gpu:
                    modalities = [m.cuda() for m in modalities]
                    target = target.cuda()
                patterns = self._group_indices_by_availability(modalities, missing_value)
                for pattern, idx in patterns.items():
                    present_models = [k for k, ok in enumerate(pattern) if ok]
                    if len(present_models) == 0:
                        continue
                    target_g = target[idx]
                    outputs_present = []
                    for mi in present_models:
                        xg = modalities[mi][idx]
                        out = self.models[mi](xg)
                        outputs_present.append(out)
                        prob = torch.softmax(out, dim=1).cpu().numpy()
                        pred = torch.argmax(out, dim=1).cpu().numpy()
                        cohort_records[mi]["y_true"].append(target_g.cpu().numpy())
                        cohort_records[mi]["y_pred"].append(pred)
                        cohort_records[mi]["y_prob"].append(prob)
                    outputs_stack = torch.stack(outputs_present)
                    weights = self._get_weights(present_models)
                    for method in self.ensemble_methods:
                        if method == "simple_average":
                            final_output = torch.mean(outputs_stack, dim=0)
                        elif method == "weighted_average":
                            if weights is None:
                                final_output = torch.mean(outputs_stack, dim=0)
                            else:
                                final_output = torch.sum(
                                    weights.unsqueeze(1).unsqueeze(2) * outputs_stack, dim=0)
                        elif method == "majority_voting":
                            final_pred = torch.mode(outputs_stack.argmax(dim=2), dim=0).values
                            final_output = F.one_hot(final_pred,
                                                      num_classes=self.num_classes).float()
                        elif method == "weighted_voting":
                            if weights is None:
                                final_pred = torch.mode(outputs_stack.argmax(dim=2), dim=0).values
                                final_output = F.one_hot(final_pred,
                                                          num_classes=self.num_classes).float()
                            else:
                                top_preds = torch.argmax(outputs_stack, dim=2)
                                nc = outputs_stack.shape[2]
                                bs = outputs_stack.shape[1]
                                wv = torch.zeros((bs, nc), device=self.device)
                                for k, mp in enumerate(top_preds):
                                    wv.scatter_add_(1, mp.unsqueeze(1),
                                        torch.full((bs, 1), float(weights[k]), device=self.device))
                                final_output = wv
                        elif method == "best_single":
                            best_model = min(
                                present_models,
                                key=lambda m: self.best_val_task_losses[m]
                                if np.isfinite(self.best_val_task_losses[m]) else float("inf")
                            )
                            j = present_models.index(best_model)
                            final_output = outputs_stack[j]
                        elif method == "greedy_ensemble":
                            chosen = [m for m in self.ens_idxs if m in present_models]
                            if len(chosen) == 0:
                                final_output = torch.mean(outputs_stack, dim=0)
                            else:
                                chosen_pos = [present_models.index(m) for m in chosen]
                                cw = self._get_weights(chosen)
                                if cw is None:
                                    final_output = torch.mean(outputs_stack[chosen_pos], dim=0)
                                else:
                                    final_output = torch.sum(
                                        cw.unsqueeze(1).unsqueeze(2)
                                        * outputs_stack[chosen_pos], dim=0)
                        else:
                            raise ValueError(method)
                        prob = torch.softmax(final_output, dim=1).cpu().numpy()
                        pred = torch.argmax(final_output, dim=1).cpu().numpy()
                        records[method]["y_true"].append(target_g.cpu().numpy())
                        records[method]["y_pred"].append(pred)
                        records[method]["y_prob"].append(prob)

        results = {}
        for method, rec in records.items():
            yt = np.concatenate(rec["y_true"])
            yp = np.concatenate(rec["y_pred"])
            yprob = np.concatenate(rec["y_prob"])
            results[method] = compute_classification_metrics(yt, yp, yprob,
                                                              num_classes=self.num_classes)
        results["cohort"] = []
        for rec in cohort_records:
            if len(rec["y_true"]) == 0:
                results["cohort"].append(None)
            else:
                yt = np.concatenate(rec["y_true"])
                yp = np.concatenate(rec["y_pred"])
                yprob = np.concatenate(rec["y_prob"])
                results["cohort"].append(
                    compute_classification_metrics(yt, yp, yprob, num_classes=self.num_classes))
        return results


# ============================================================
# META FUSION TRAINER (rho_search only -- no ablation)
# ============================================================


class TrainerMetaFusionMetrics(Trainer_new):
    """Patched test_classification that collects per-cohort probabilities for
    binary AUC. Trains via rho_search; ablation NOT used here per user request."""

    def test_classification(self, ensemble_methods, test_loader, best_val_task_losses):
        records = {m: {"y_true": [], "y_pred": [], "y_prob": []} for m in ensemble_methods}
        cohort_records = [{"y_true": [], "y_pred": [], "y_prob": []} for _ in range(self.model_num)]
        for i in range(self.model_num):
            self.models[i].eval()
        if "meta_learner" in ensemble_methods:
            self.meta_learner.eval()
        with torch.no_grad():
            for batch in test_loader:
                modalities, target = batch[:-1], batch[-1]
                if self.use_gpu:
                    modalities = [mod.cuda() for mod in modalities]
                    target = target.cuda()
                outputs = []
                for i, model in enumerate(self.models):
                    output = model(modalities[i])
                    outputs.append(output)
                    prob = torch.softmax(output, dim=1).cpu().numpy()
                    pred = torch.argmax(output, dim=1).cpu().numpy()
                    cohort_records[i]["y_true"].append(target.cpu().numpy())
                    cohort_records[i]["y_pred"].append(pred)
                    cohort_records[i]["y_prob"].append(prob)
                outputs_stack = torch.stack(outputs)

                for method in ensemble_methods:
                    if method == "simple_average":
                        final_output = torch.mean(outputs_stack, dim=0)
                    elif method == "weighted_average":
                        weights = get_weights_by_task_loss(best_val_task_losses).to(outputs_stack.device)
                        final_output = torch.sum(weights.unsqueeze(1).unsqueeze(2) * outputs_stack, dim=0)
                    elif method == "majority_voting":
                        final_pred = torch.mode(outputs_stack.argmax(dim=2), dim=0).values
                        final_output = F.one_hot(final_pred, num_classes=self.num_classes).float()
                    elif method == "weighted_voting":
                        weights = get_weights_by_task_loss(best_val_task_losses).to(outputs_stack.device)
                        top_preds = torch.argmax(outputs_stack, dim=2)
                        nc = outputs_stack.shape[2]
                        bs = outputs_stack.shape[1]
                        wv = torch.zeros((bs, nc), device=outputs_stack.device)
                        for k, mp in enumerate(top_preds):
                            wv.scatter_add_(1, mp.unsqueeze(1),
                                torch.full((bs, 1), float(weights[k]), device=outputs_stack.device))
                        final_output = wv
                    elif method == "meta_learner":
                        outputs_concat = torch.cat(outputs, dim=1)
                        final_output = self.meta_learner(outputs_concat)
                    elif method == "best_single":
                        best_model = best_val_task_losses.index(min(best_val_task_losses))
                        final_output = outputs_stack[best_model]
                    elif method == "greedy_ensemble":
                        weights = get_weights_by_task_loss(best_val_task_losses)[self.ens_idxs].to(outputs_stack.device)
                        weights = weights / torch.sum(weights)
                        final_output = torch.sum(
                            weights.unsqueeze(1).unsqueeze(2) * outputs_stack[self.ens_idxs], dim=0)
                    else:
                        raise ValueError(method)
                    prob = torch.softmax(final_output, dim=1).cpu().numpy()
                    pred = torch.argmax(final_output, dim=1).cpu().numpy()
                    records[method]["y_true"].append(target.cpu().numpy())
                    records[method]["y_pred"].append(pred)
                    records[method]["y_prob"].append(prob)

        results = {}
        for method, rec in records.items():
            yt = np.concatenate(rec["y_true"])
            yp = np.concatenate(rec["y_pred"])
            yprob = np.concatenate(rec["y_prob"])
            results[method] = compute_classification_metrics(yt, yp, yprob,
                                                              num_classes=self.num_classes)
        results["cohort"] = []
        for rec in cohort_records:
            yt = np.concatenate(rec["y_true"])
            yp = np.concatenate(rec["y_pred"])
            yprob = np.concatenate(rec["y_prob"])
            results["cohort"].append(
                compute_classification_metrics(yt, yp, yprob, num_classes=self.num_classes))
        return results


# ============================================================
# JOINT TRAINER (existing pattern)
# ============================================================


class TrainerJointTCGA(Trainer_Joint_new):
    def __init__(self, config, models, data_loaders):
        super().__init__(config, models, data_loaders)
        self.loss_mse = self.loss_task

    def test(self, test_loader, missing_value=None):
        self.missing_value = missing_value
        if "meta_learner" in self.ensemble_methods:
            self.meta_learner = self.initialize_meta_learner()
            self.meta_learner = self.meta_learner.to(self.device)
            self.meta_learner_optimizer = torch.optim.Adam(
                self.meta_learner.parameters(), lr=0.1, weight_decay=0)
            self.load_meta_learner()
        if self.task_type == "classification":
            best_val_task_losses = self.validate(missing_value=getattr(self, "missing_value", None))
            best_val_task_losses = [best_val_task_losses[i].avg for i in range(self.model_num)]
            return self.test_classification_metrics(
                self.ensemble_methods + ["best_single"], test_loader, best_val_task_losses)
        else:
            return super().test(test_loader, missing_value=missing_value)

    def test_classification_metrics(self, ensemble_methods, test_loader, best_val_task_losses):
        records = {m: {"y_true": [], "y_pred": [], "y_prob": []} for m in ensemble_methods}
        cohort_records = [{"y_true": [], "y_pred": [], "y_prob": []} for _ in range(self.model_num)]
        for i in range(self.model_num):
            self.models[i].eval()
        if "meta_learner" in ensemble_methods:
            self.meta_learner.eval()

        with torch.no_grad():
            for batch in test_loader:
                modalities, target = batch[:-1], batch[-1]
                if self.use_gpu:
                    modalities = [m.cuda() for m in modalities]
                    target = target.cuda()
                missing_value = getattr(self, "missing_value", None)
                weights_all = get_weights_by_task_loss(best_val_task_losses)

                if missing_value is None:
                    outputs = []
                    for i, model in enumerate(self.models):
                        output = model(modalities[i])
                        outputs.append(output)
                        prob = torch.softmax(output, dim=1).cpu().numpy()
                        pred = torch.argmax(output, dim=1).cpu().numpy()
                        cohort_records[i]["y_true"].append(target.cpu().numpy())
                        cohort_records[i]["y_pred"].append(pred)
                        cohort_records[i]["y_prob"].append(prob)
                    outputs_stack = torch.stack(outputs)

                    for method in ensemble_methods:
                        if method == "simple_average":
                            final_output = torch.mean(outputs_stack, dim=0)
                        elif method == "weighted_average":
                            final_output = torch.sum(
                                weights_all.to(outputs_stack.device).unsqueeze(1).unsqueeze(2)
                                * outputs_stack, dim=0)
                        elif method == "majority_voting":
                            final_pred = torch.mode(outputs_stack.argmax(dim=2), dim=0).values
                            final_output = F.one_hot(final_pred,
                                                      num_classes=self.num_classes).float()
                        elif method == "weighted_voting":
                            top_preds = torch.argmax(outputs_stack, dim=2)
                            nc = outputs_stack.shape[2]
                            bs = outputs_stack.shape[1]
                            wv = torch.zeros((bs, nc), device=outputs_stack.device)
                            for k, mp in enumerate(top_preds):
                                wv.scatter_add_(1, mp.unsqueeze(1),
                                    torch.full((bs, 1), float(weights_all[k]),
                                                device=outputs_stack.device))
                            final_output = wv
                        elif method == "meta_learner":
                            outputs_concat = torch.cat(outputs, dim=1)
                            final_output = self.meta_learner(outputs_concat)
                        elif method == "best_single":
                            best_model = best_val_task_losses.index(min(best_val_task_losses))
                            final_output = outputs_stack[best_model]
                        elif method == "greedy_ensemble":
                            weights = get_weights_by_task_loss(best_val_task_losses)[self.ens_idxs]
                            weights = weights / torch.sum(weights)
                            weights = weights.unsqueeze(1).unsqueeze(2).to(outputs_stack.device)
                            final_output = torch.sum(weights * outputs_stack[self.ens_idxs], dim=0)
                        else:
                            raise ValueError(method)
                        prob = torch.softmax(final_output, dim=1).cpu().numpy()
                        pred = torch.argmax(final_output, dim=1).cpu().numpy()
                        records[method]["y_true"].append(target.cpu().numpy())
                        records[method]["y_pred"].append(pred)
                        records[method]["y_prob"].append(prob)
                else:
                    patterns = self._group_indices_by_availability(modalities, missing_value)
                    for pattern, idx in patterns.items():
                        present_models = [k for k, ok in enumerate(pattern) if ok]
                        if len(present_models) == 0:
                            continue
                        mods_g = [modalities[k][idx] for k in range(self.model_num)]
                        target_g = target[idx]
                        outputs_present = []
                        for mi in present_models:
                            out = self.models[mi](mods_g[mi])
                            outputs_present.append(out)
                            prob = torch.softmax(out, dim=1).cpu().numpy()
                            pred = torch.argmax(out, dim=1).cpu().numpy()
                            cohort_records[mi]["y_true"].append(target_g.cpu().numpy())
                            cohort_records[mi]["y_pred"].append(pred)
                            cohort_records[mi]["y_prob"].append(prob)
                        outputs_stack = torch.stack(outputs_present)

                        for method in ensemble_methods:
                            if method == "simple_average":
                                final_output = torch.mean(outputs_stack, dim=0)
                            elif method == "weighted_average":
                                w = weights_all[present_models]
                                w = w / torch.sum(w)
                                final_output = torch.sum(
                                    w.to(outputs_stack.device).unsqueeze(1).unsqueeze(2)
                                    * outputs_stack, dim=0)
                            elif method == "majority_voting":
                                final_pred = torch.mode(outputs_stack.argmax(dim=2), dim=0).values
                                final_output = F.one_hot(final_pred,
                                                          num_classes=self.num_classes).float()
                            elif method == "weighted_voting":
                                w = weights_all[present_models]
                                w = w / torch.sum(w)
                                top_preds = torch.argmax(outputs_stack, dim=2)
                                nc = outputs_stack.shape[2]
                                bs = outputs_stack.shape[1]
                                wv = torch.zeros((bs, nc), device=outputs_stack.device)
                                for k, mp in enumerate(top_preds):
                                    wv.scatter_add_(1, mp.unsqueeze(1),
                                        torch.full((bs, 1), float(w[k]),
                                                    device=outputs_stack.device))
                                final_output = wv
                            elif method == "meta_learner":
                                full_outputs = []
                                for mi in range(self.model_num):
                                    if mi in present_models:
                                        j = present_models.index(mi)
                                        full_outputs.append(outputs_present[j])
                                    else:
                                        full_outputs.append(torch.zeros_like(outputs_present[0]))
                                outputs_concat = torch.cat(full_outputs, dim=1)
                                final_output = self.meta_learner(outputs_concat)
                            elif method == "best_single":
                                best_model = min(present_models,
                                                  key=lambda m: best_val_task_losses[m])
                                j = present_models.index(best_model)
                                final_output = outputs_stack[j]
                            elif method == "greedy_ensemble":
                                chosen = [m for m in getattr(self, "ens_idxs",
                                                              list(range(self.model_num)))
                                          if m in present_models]
                                if len(chosen) == 0:
                                    final_output = torch.mean(outputs_stack, dim=0)
                                else:
                                    w = weights_all[chosen]
                                    w = w / torch.sum(w)
                                    chosen_pos = [present_models.index(m) for m in chosen]
                                    final_output = torch.sum(
                                        w.to(outputs_stack.device).unsqueeze(1).unsqueeze(2)
                                        * outputs_stack[chosen_pos], dim=0)
                            else:
                                raise ValueError(method)
                            prob = torch.softmax(final_output, dim=1).cpu().numpy()
                            pred = torch.argmax(final_output, dim=1).cpu().numpy()
                            records[method]["y_true"].append(target_g.cpu().numpy())
                            records[method]["y_pred"].append(pred)
                            records[method]["y_prob"].append(prob)

        results = {}
        for method, rec in records.items():
            yt = np.concatenate(rec["y_true"])
            yp = np.concatenate(rec["y_pred"])
            yprob = np.concatenate(rec["y_prob"])
            results[method] = compute_classification_metrics(yt, yp, yprob,
                                                              num_classes=self.num_classes)
        results["cohort"] = []
        for rec in cohort_records:
            if len(rec["y_true"]) == 0:
                results["cohort"].append(None)
            else:
                yt = np.concatenate(rec["y_true"])
                yp = np.concatenate(rec["y_pred"])
                yprob = np.concatenate(rec["y_prob"])
                results["cohort"].append(
                    compute_classification_metrics(yt, yp, yprob, num_classes=self.num_classes))
        return results


# ============================================================
# RESULT HELPERS
# ============================================================


def flatten_results(family, training_mode, setting_name, results, repetition, split_seed):
    rows = []
    for k, v in results.items():
        if k == "cohort":
            for i, item in enumerate(v):
                if item is None:
                    continue
                if isinstance(item, dict):
                    rows.append({"repetition": repetition, "split_seed": split_seed,
                                 "family": family, "training_mode": training_mode,
                                 "setting": setting_name, "method": f"cohort_{i}", **item})
                else:
                    rows.append({"repetition": repetition, "split_seed": split_seed,
                                 "family": family, "training_mode": training_mode,
                                 "setting": setting_name, "method": f"cohort_{i}",
                                 "value": float(item)})
        else:
            if isinstance(v, dict):
                rows.append({"repetition": repetition, "split_seed": split_seed,
                             "family": family, "training_mode": training_mode,
                             "setting": setting_name, "method": k, **v})
            else:
                rows.append({"repetition": repetition, "split_seed": split_seed,
                             "family": family, "training_mode": training_mode,
                             "setting": setting_name, "method": k, "value": float(v)})
    return rows


SETTING_ORDER = [
    "filtered",
    "imputation_filteredpart",
    "imputation_extrapart",
    "imputation_overall",
]


def apply_setting_order(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["setting"] = pd.Categorical(df["setting"], categories=SETTING_ORDER, ordered=True)
    sort_cols = [c for c in ["repetition", "split_seed", "setting", "family",
                              "training_mode", "method"] if c in df.columns]
    df = df.sort_values(sort_cols).reset_index(drop=True)
    return df


def compute_standard_error(series: pd.Series) -> float:
    x = series.dropna().astype(float)
    n = len(x)
    if n <= 1:
        return np.nan
    return float(x.std(ddof=1) / np.sqrt(n))


def average_results_over_repetitions(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    group_cols = ["family", "training_mode", "setting", "method"]
    metric_candidates = ["accuracy", "macro_f1", "auc_ovr", "sensitivity",
                          "specificity", "value"]
    metric_cols = [c for c in metric_candidates if c in df.columns]
    agg_dict = {}
    for metric in metric_cols:
        agg_dict[f"{metric}_mean"] = (metric, "mean")
        agg_dict[f"{metric}_se"] = (metric, compute_standard_error)
    avg_df = (df.groupby(group_cols, dropna=False, observed=True)
                 .agg(**agg_dict).reset_index())
    rep_counts = (df.groupby(group_cols, dropna=False, observed=True)["repetition"]
                    .nunique().reset_index(name="num_repetitions"))
    avg_df = avg_df.merge(rep_counts, on=group_cols, how="left")
    avg_df = apply_setting_order(avg_df)
    return avg_df


# ============================================================
# 3-WAY EVALUATION HELPER
# ============================================================


def evaluate_three_way_partition(model_obj, family_name, training_mode, setting,
                                  all_rows, repetition, split_seed,
                                  prefix, needs_missing_value=False):
    full_loader, filt_loader, extra_loader = make_partitioned_test_loaders(setting, BATCH_SIZE)
    if needs_missing_value:
        mv = setting["missing_value"]
        res_filt = model_obj.test(filt_loader, missing_value=mv)
        all_rows.extend(flatten_results(family_name, training_mode,
                                         f"{prefix}_filteredpart", res_filt, repetition, split_seed))
        res_extra = model_obj.test(extra_loader, missing_value=mv)
        all_rows.extend(flatten_results(family_name, training_mode,
                                         f"{prefix}_extrapart", res_extra, repetition, split_seed))
        res_all = model_obj.test(full_loader, missing_value=mv)
        all_rows.extend(flatten_results(family_name, training_mode,
                                         f"{prefix}_overall", res_all, repetition, split_seed))
    else:
        res_filt = model_obj.test(filt_loader)
        all_rows.extend(flatten_results(family_name, training_mode,
                                         f"{prefix}_filteredpart", res_filt, repetition, split_seed))
        res_extra = model_obj.test(extra_loader)
        all_rows.extend(flatten_results(family_name, training_mode,
                                         f"{prefix}_extrapart", res_extra, repetition, split_seed))
        res_all = model_obj.test(full_loader)
        all_rows.extend(flatten_results(family_name, training_mode,
                                         f"{prefix}_overall", res_all, repetition, split_seed))


# ============================================================
# SETTINGS BUILDER
# ============================================================


def build_all_settings(split_seed,
                        filt_patient_ids, filt_y, filt_arrays,
                        full_patient_ids, full_y, full_arrays):
    """Build filtered + imputation 3-way settings for one repetition."""
    print(f"\nBuilding settings for split_seed={split_seed}")

    # ---- Filtered setting ----
    f_train, f_val, f_test = build_patient_level_splits(
        filt_patient_ids, filt_y,
        test_size=TEST_SIZE, val_size_within_train=VAL_SIZE_WITHIN_TRAIN,
        random_state=split_seed,
    )
    f_train_arr_raw, f_y_train = subset_by_indices(filt_arrays, filt_y, f_train)
    f_val_arr_raw, f_y_val = subset_by_indices(filt_arrays, filt_y, f_val)
    f_test_arr_raw, f_y_test = subset_by_indices(filt_arrays, filt_y, f_test)
    f_train_arr, f_val_arr, f_test_arr = standardize_per_modality(
        f_train_arr_raw, f_val_arr_raw, f_test_arr_raw)

    print(f"  filtered split: train={len(f_y_train)}, val={len(f_y_val)}, test={len(f_y_test)}")

    # ---- Imputation 3-way setting ----
    imp_split = build_imputation_splits(
        full_patient_ids, full_y, full_arrays, MISSING_VALUE,
        test_size=TEST_SIZE, val_size_within_train=VAL_SIZE_WITHIN_TRAIN,
        random_state=split_seed,
    )
    i_train_arr_raw, i_y_train = subset_by_indices(full_arrays, full_y, imp_split["imp_train_idx"])
    i_val_arr_raw, i_y_val = subset_by_indices(full_arrays, full_y, imp_split["imp_val_idx"])
    i_test_arr_raw, i_y_test = subset_by_indices(full_arrays, full_y, imp_split["imp_test_idx"])
    i_train_arr, i_val_arr, i_test_arr = standardize_per_modality_with_sentinel(
        i_train_arr_raw, i_val_arr_raw, i_test_arr_raw, MISSING_VALUE)

    print(f"  imputation split: train={len(i_y_train)}, val={len(i_y_val)}, "
          f"test={len(i_y_test)} (filteredpart={imp_split['test_filtered_mask'].sum()}, "
          f"extrapart={imp_split['test_extra_mask'].sum()})")

    return {
        "filtered": {
            "train_arrays": f_train_arr, "val_arrays": f_val_arr, "test_arrays": f_test_arr,
            "y_train": f_y_train, "y_val": f_y_val, "y_test": f_y_test,
            "missing_value": None,
        },
        "imputation": {
            "train_arrays": i_train_arr, "val_arrays": i_val_arr, "test_arrays": i_test_arr,
            "y_train": i_y_train, "y_val": i_y_val, "y_test": i_y_test,
            "missing_value": MISSING_VALUE,
            "test_filtered_mask": imp_split["test_filtered_mask"],
            "test_extra_mask": imp_split["test_extra_mask"],
        },
    }


# ============================================================
# ONE-REP EXPERIMENT
# ============================================================


def run_full_experiment_one_seed(repetition, split_seed,
                                  filt_patient_ids, filt_y, filt_arrays,
                                  full_patient_ids, full_y, full_arrays):
    print("\n" + "#" * 100)
    print(f"REPETITION {repetition + 1}/{NUM_REPETITIONS} | split_seed={split_seed}")
    print("#" * 100)

    seed_everything(split_seed)
    config = get_config_for_seed(split_seed)
    all_rows = []

    settings = build_all_settings(
        split_seed,
        filt_patient_ids, filt_y, filt_arrays,
        full_patient_ids, full_y, full_arrays,
    )

    # ============ FILTERED SETTING ============
    f_set = settings["filtered"]
    train_loader, val_loader, test_loader = make_loaders_from_arrays(
        f_set["train_arrays"], f_set["val_arrays"], f_set["test_arrays"],
        f_set["y_train"], f_set["y_val"], f_set["y_test"], BATCH_SIZE,
    )
    input_dims_f = [a.shape[1] for a in f_set["train_arrays"]]
    mv_f = f_set["missing_value"]

    # 1) Benchmarks
    print("\n--- [filtered] Benchmarks ---")
    bm = Benchmarks(config, build_benchmark_models(input_dims_f),
                     [train_loader, val_loader], model_dims=None)
    bm.train()
    bm_res = bm.test(test_loader)
    all_rows.extend(flatten_results("benchmarks", "na", "filtered", bm_res, repetition, split_seed))

    # 2) Late fusion all-ensembles (option 1; on filtered there's no missingness)
    print("\n--- [filtered] LateFusion all-ensembles ---")
    lf = LateFusionAllEnsemblesBenchmark(config, build_models(input_dims_f))
    lf.train(train_loader, val_loader)
    lf_res = lf.test(test_loader)
    all_rows.extend(flatten_results("late_fusion", "all_ensembles", "filtered",
                                     lf_res, repetition, split_seed))

    # 3) Joint marginal
    # NOTE: ordering kept identical to the previous tcga_brca_survival_train.py
    # (benchmarks -> late_fusion -> joint_marginal -> joint_shapley) so that
    # the RNG state seen by these trainers is unchanged. MetaFusion below
    # is run AFTER the joints to avoid shifting their RNG state.
    print("\n--- [filtered] Joint marginal ---")
    jm = TrainerJointTCGA(config, build_models(input_dims_f), [train_loader, val_loader])
    jm.train("marginal", missing_value=mv_f)
    jm_res = jm.test(test_loader, missing_value=mv_f)
    all_rows.extend(flatten_results("joint", "marginal", "filtered",
                                     jm_res, repetition, split_seed))

    # 4) Joint shapley
    print("\n--- [filtered] Joint shapley ---")
    js = TrainerJointTCGA(config, build_models(input_dims_f), [train_loader, val_loader])
    js.train("shapley", missing_value=mv_f)
    js_res = js.test(test_loader, missing_value=mv_f)
    all_rows.extend(flatten_results("joint", "shapley", "filtered",
                                     js_res, repetition, split_seed))

    # 5) Meta fusion (rho_search only -- no ablation).
    # Placed AFTER the joint trainers so it does not shift the RNG state
    # they consume during model init and DataLoader shuffling.
    print("\n--- [filtered] MetaFusion rho_search ---")
    mf = TrainerMetaFusionMetrics(config, build_models(input_dims_f),
                                    [train_loader, val_loader])
    mf.train()
    mf_res = mf.test(test_loader)
    all_rows.extend(flatten_results("metafusion", "rho_search", "filtered",
                                     mf_res, repetition, split_seed))

    # ============ IMPUTATION 3-WAY SETTING ============
    i_set = settings["imputation"]
    train_loader_i, val_loader_i, test_loader_i = make_loaders_from_arrays(
        i_set["train_arrays"], i_set["val_arrays"], i_set["test_arrays"],
        i_set["y_train"], i_set["y_val"], i_set["y_test"], BATCH_SIZE,
    )
    input_dims_i = [a.shape[1] for a in i_set["train_arrays"]]
    mv_i = i_set["missing_value"]

    # 1) Benchmarks (3-way eval)
    print("\n--- [imputation] Benchmarks ---")
    bm = Benchmarks(config, build_benchmark_models(input_dims_i),
                     [train_loader_i, val_loader_i], model_dims=None)
    bm.train()
    evaluate_three_way_partition(
        bm, "benchmarks", "na", i_set,
        all_rows, repetition, split_seed,
        prefix="imputation", needs_missing_value=False,
    )

    # 2) Late fusion all-ensembles (option 1, 3-way eval)
    print("\n--- [imputation] LateFusion all-ensembles (option 1) ---")
    lf = LateFusionAllEnsemblesBenchmark(config, build_models(input_dims_i))
    lf.train(train_loader_i, val_loader_i)
    evaluate_three_way_partition(
        lf, "late_fusion", "all_ensembles", i_set,
        all_rows, repetition, split_seed,
        prefix="imputation", needs_missing_value=False,
    )

    # 3) Late fusion available-only (option 2, 3-way eval)
    print("\n--- [imputation] LateFusion available-only (option 2) ---")
    lfa = LateFusionAvailableOnlyBenchmark(config, input_dims_i)
    lfa.train(train_arrays=i_set["train_arrays"], val_arrays=i_set["val_arrays"],
               y_train=i_set["y_train"], y_val=i_set["y_val"])
    evaluate_three_way_partition(
        lfa, "late_fusion_avail", "available_only", i_set,
        all_rows, repetition, split_seed,
        prefix="imputation", needs_missing_value=True,
    )

    # 4) Meta fusion (rho_search, 3-way eval)
    print("\n--- [imputation] MetaFusion rho_search ---")
    mf = TrainerMetaFusionMetrics(config, build_models(input_dims_i),
                                    [train_loader_i, val_loader_i])
    mf.train()
    evaluate_three_way_partition(
        mf, "metafusion", "rho_search", i_set,
        all_rows, repetition, split_seed,
        prefix="imputation", needs_missing_value=False,
    )

    # 5) Joint marginal (3-way eval, with missing_value)
    print("\n--- [imputation] Joint marginal ---")
    jm = TrainerJointTCGA(config, build_models(input_dims_i),
                           [train_loader_i, val_loader_i])
    jm.train("marginal", missing_value=mv_i)
    evaluate_three_way_partition(
        jm, "joint", "marginal", i_set,
        all_rows, repetition, split_seed,
        prefix="imputation", needs_missing_value=True,
    )

    # 6) Joint shapley (3-way eval, with missing_value)
    print("\n--- [imputation] Joint shapley ---")
    js = TrainerJointTCGA(config, build_models(input_dims_i),
                           [train_loader_i, val_loader_i])
    js.train("shapley", missing_value=mv_i)
    evaluate_three_way_partition(
        js, "joint", "shapley", i_set,
        all_rows, repetition, split_seed,
        prefix="imputation", needs_missing_value=True,
    )

    rep_df = pd.DataFrame(all_rows)
    rep_df = apply_setting_order(rep_df)
    print(f"\nFinished repetition {repetition + 1}, rows: {len(rep_df)}")
    return rep_df


# ============================================================
# MAIN
# ============================================================


def run_repeated_experiments():
    # Load both cohorts once -- only the train/val/test split changes per rep.
    filt_pids, filt_y, filt_arrs = load_filtered_cohort_with_survival(
        DATA_DIR, RAW_DATA_DIR, SURVIVAL_THRESHOLD_DAYS)
    full_pids, full_y, full_arrs = load_full_cohort_with_survival(
        DATA_DIR, RAW_DATA_DIR, SURVIVAL_THRESHOLD_DAYS)

    all_rep_dfs = []
    for rep_idx, split_seed in enumerate(REPETITION_SEEDS):
        rep_df = run_full_experiment_one_seed(
            repetition=rep_idx, split_seed=split_seed,
            filt_patient_ids=filt_pids, filt_y=filt_y, filt_arrays=filt_arrs,
            full_patient_ids=full_pids, full_y=full_y, full_arrays=full_arrs,
        )
        all_rep_dfs.append(rep_df)
        partial = pd.concat(all_rep_dfs, axis=0, ignore_index=True)
        partial = apply_setting_order(partial)
        partial.to_csv(
            os.path.join(OUTPUT_DIR, "tcga_brca_survival_4settings_partial.csv"), index=False)

    raw_df = pd.concat(all_rep_dfs, axis=0, ignore_index=True)
    avg_df = average_results_over_repetitions(raw_df)
    raw_df = apply_setting_order(raw_df)
    avg_df = apply_setting_order(avg_df)
    return raw_df, avg_df


if __name__ == "__main__":
    seed_everything(RANDOM_STATE)
    print("=" * 80)
    print("TCGA-BRCA filtered + imputation 3-way, OVERALL SURVIVAL, 20 reps")
    print("=" * 80)
    print(f"  USE_GPU              = {USE_GPU}")
    print(f"  SURVIVAL_THRESHOLD   = {SURVIVAL_THRESHOLD_DAYS} days "
          f"({SURVIVAL_THRESHOLD_DAYS/365:.1f} years)")
    print(f"  NUM_REPETITIONS      = {NUM_REPETITIONS}")
    print(f"  REPETITION_SEEDS     = {REPETITION_SEEDS}")
    print(f"  EPOCHS               = {EPOCHS}")
    print(f"  BATCH_SIZE           = {BATCH_SIZE}")
    print(f"  HIDDEN_DIMS          = {HIDDEN_DIMS}")
    print(f"  NUM_CLASSES          = {NUM_CLASSES} (binary)")
    print(f"  MISSING_VALUE        = {MISSING_VALUE}")

    raw_df, avg_df = run_repeated_experiments()

    raw_out = os.path.join(OUTPUT_DIR, "tcga_brca_survival_4settings_20reps_raw.csv")
    avg_out = os.path.join(OUTPUT_DIR, "tcga_brca_survival_4settings_20reps_avg_with_se.csv")
    raw_df.to_csv(raw_out, index=False)
    avg_df.to_csv(avg_out, index=False)

    print("\nRAW HEAD:")
    print(raw_df.head(10))
    print("\nAVG HEAD:")
    print(avg_df.head(10))
    print(f"\nSaved raw:     {raw_out}")
    print(f"Saved avg:     {avg_out}")

/home/zmoslemi/.local/lib/python3.10/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: libc10_cuda.so: cannot open shared object file: No such file or directory
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/home/zmoslemi/.local/lib/python3.10/site-packages/torch_geometric/typing.py:124: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: libc10_cuda.so: cannot open shared object file: No such file or directory
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "
Disabling PyTorch because PyTorch >= 2.4 is required but found 2.3.1+cpu
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
I0000 00:00:1777782071.043679 3062725 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-

TCGA-BRCA filtered + imputation 3-way, OVERALL SURVIVAL, 20 reps
  USE_GPU              = False
  SURVIVAL_THRESHOLD   = 1825 days (5.0 years)
  NUM_REPETITIONS      = 20
  REPETITION_SEEDS     = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61]
  EPOCHS               = 100
  BATCH_SIZE           = 64
  HIDDEN_DIMS          = [256, 128]
  NUM_CLASSES          = 2 (binary)
  MISSING_VALUE        = -999.0
Loading FILTERED cohort + survival
  filtered cohort (PAM50 step): 529 patients
  drop: no_survival=5, censored_early=328, keep=196
  Filtered final: 196 patients  (alive=148 75.5%, died=48 24.5%)
  shapes: mRNA (196, 2000), miRNA (196, 1881), methyl (196, 5000)
Loading FULL cohort + survival
  full cohort (PAM50 step): 842 patients
  drop: no_survival=17, censored_early=527, keep=298
  missing rows per modality: mRNA=0, miRNA=6, methyl=98
  Full final: 298 patients  (alive=222 74.5%, died=76 25.5%)

########################################################

100%|███████████████████████████████████████| 124/124 [00:00<00:00, 1113.19it/s]


model_1: train loss: 0.708, train task loss: 0.708 - val loss: 0.584, val task loss: 0.584 [*] Best so far
model_2: train loss: 0.666, train task loss: 0.666 - val loss: 0.606, val task loss: 0.606 [*] Best so far
model_3: train loss: 0.660, train task loss: 0.660 - val loss: 0.693, val task loss: 0.693 [*] Best so far
model_4: train loss: 0.652, train task loss: 0.652 - val loss: 0.663, val task loss: 0.663 [*] Best so far

Epoch: 2/100 - LR: 0.001000


100%|███████████████████████████████████████| 124/124 [00:00<00:00, 1193.92it/s]

...

model_1: train loss: 0.000, train task loss: 0.000 - val loss: 1.263, val task loss: 1.263
model_2: train loss: 7.180, train task loss: 7.180 - val loss: 1.007, val task loss: 1.007
model_3: train loss: 62.763, train task loss: 62.763 - val loss: 50.883, val task loss: 50.883
model_4: train loss: 22.569, train task loss: 22.569 - val loss: 26.697, val task loss: 26.697

Epoch: 39/100 - LR: 0.001000


 68%|██████████████████████████▌            | 128/188 [00:00<00:00, 1921.29it/s]


model_1: train loss: 0.000, train task loss: 0.000 - val loss: 1.263, val task loss: 1.263
model_2: train loss: 17.663, train task loss: 17.663 - val loss: 1.016, val task loss: 1.016
model_3: train loss: 54.801, train task loss: 54.801 - val loss: 101.740, val task loss: 101.740
model_4: train loss: 19.766, train task loss: 19.766 - val loss: 7.114, val task loss: 7.114 [*] Best so far

Epoch: 40/100 - LR: 0.001000


 34%|█████████████▉                           | 64/188 [00:00<00:00, 838.79it/s]


KeyboardInterrupt: 